# Experiment 5: Data Validation and Reproducible ML Pipelines

## AIM:
Design and validate schema-aware data preprocessing workflows by identifying datatype mismatches, schema inconsistencies, missing attributes, invalid categorical values, and corrupted records. Build reusable and idempotent preprocessing pipelines and validate transformed features, preprocessing outputs, and dataset consistency across repeated executions.

---
### Objectives:
1. **Schema Definition**: Define a strict `Pandera` DataFrameSchema for the raw Walmart dataset.
2. **Anomaly & Corruption Detection**: Inject controlled corruptions (negative sales, invalid date formats, nulls in mandatory columns, out-of-range categorical flags) and capture schema error reports.
3. **Data Quarantining**: Implement an automated quarantine routing mechanism to isolate bad records from clean training data.
4. **Idempotent Preprocessing Pipeline**: Build custom Scikit-Learn transformers and ColumnTransformers for reproducible feature engineering.
5. **Idempotency Assertions**: Verify that running the transform pipeline multiple times yields bit-for-bit identical feature matrices without numerical drift.


In [ ]:
import os
import sys
import numpy as np
import pandas as pd

os.environ["DISABLE_PANDERA_IMPORT_WARNING"] = "True"
try:
    import pandera.pandas as pa
except ImportError:
    import pandera as pa

if ".." not in sys.path:
    sys.path.insert(0, "..")

from src.data_validation import (
    get_raw_data_schema,
    get_transformed_features_schema,
    validate_dataset,
    generate_corrupted_samples,
    quarantine_corrupted_records
)
from src.pipeline import build_preprocessor, verify_pipeline_idempotency
print("Pandera and Validation modules loaded successfully. Pandera version:", pa.__version__)


## Step 1: Validate Clean Dataset Against Pandera Schema
We define declarative validation rules for all 8 columns:
- `Store`: integer between 1 and 100
- `Date`: valid parseable `DD-MM-YYYY` string
- `Weekly_Sales`: float $\ge 0.0$
- `Holiday_Flag`: binary categorical in $\{0, 1\}$
- `Temperature`: valid temperature range $[-50^\circ	ext{F}, 150^\circ	ext{F}]$
- `Fuel_Price`, `CPI`: strictly positive floats $> 0$
- `Unemployment`: non-negative float $\ge 0$


In [ ]:
raw_df = pd.read_csv(os.path.join("..", "Walmart.csv"))
print(f"Loaded raw data: {raw_df.shape[0]} rows, {raw_df.shape[1]} columns")

is_valid, validated_df, failures = validate_dataset(raw_df, lazy=True)
print(f"Dataset Validation Status: {'PASSED' if is_valid else 'FAILED'}")
if is_valid:
    print("All records satisfy schema data types, nullability, and boundary constraints.")


## Step 2: Inject Controlled Data Corruptions
In real-world MLOps pipelines, upstream data ingestion can suffer from corruption. We create synthetic bad records to verify our validation suite's robustness:
1. Negative sales value (`-$125,000.50`)
2. Corrupted date string (`2020/99/99_INVALID`)
3. Out-of-bounds categorical value (`Holiday_Flag = 99`)
4. Physically impossible temperature (`-105°F`)
5. String datatype in numeric fuel price (`"three_dollars"`)
6. Missing/NaN in mandatory unemployment rate


In [ ]:
corrupted_sample = generate_corrupted_samples(raw_df)
display(corrupted_sample)


## Step 3: Capture Schema Violation Audit Report
We run lazy validation across the corrupted records. Pandera catches all violations in a single pass without halting execution.


In [ ]:
is_corrupt_valid, _, failure_report = validate_dataset(corrupted_sample, lazy=True)
print(f"Validation Result on Corrupted Data: is_valid = {is_corrupt_valid}")

if failure_report is not None:
    cols = [c for c in ["schema_context", "column", "check", "failure_case", "index"] if c in failure_report.columns]
    print("
=== PANDERA SCHEMA FAILURE AUDIT REPORT ===")
    display(failure_report[cols].drop_duplicates())


## Step 4: Row-Level Data Quarantining Workflow
Rather than crashing downstream batch scoring, corrupted records are isolated into a quarantined audit partition while clean records flow safely into model preprocessing.


In [ ]:
mixed_dataset = pd.concat([corrupted_sample, raw_df.head(200)], ignore_index=True)
clean_partition, quarantined_partition, audit = quarantine_corrupted_records(mixed_dataset)

print(f"Total Records Ingested : {audit['total_records']}")
print(f"Clean Records Accepted : {audit['clean_records']}")
print(f"Quarantined Records    : {audit['quarantined_records']} ({audit['quarantine_rate_percent']}%)")

print("
Quarantined Records Audit Sample:")
display(quarantined_partition[["Store", "Date", "Weekly_Sales", "quarantine_reason"]])


## Step 5: Preprocessing Pipeline & Feature Schema Validation
We pass the clean partition through our reusable Scikit-Learn preprocessing pipeline (`DateFeatureExtractor` + `ColumnTransformer` + `StandardScaler`).


In [ ]:
preprocessor = build_preprocessor(scaler_type="standard", with_feature_selection=False)

X_clean = clean_partition.drop(columns=["Weekly_Sales"])
y_clean = clean_partition["Weekly_Sales"]

preprocessor.fit(X_clean, y_clean)
transformed_features = preprocessor.transform(X_clean)

print(f"Raw Input Feature Shape        : {X_clean.shape}")
print(f"Transformed Output Matrix Shape: {transformed_features.shape}")

# Validate preprocessed output with TransformedFeaturesSchema
feature_schema = get_transformed_features_schema(num_expected_features=transformed_features.shape[1])
transformed_df = pd.DataFrame(
    transformed_features,
    columns=[f"feature_{i}" for i in range(transformed_features.shape[1])]
)

feature_schema.validate(transformed_df)
print(">>> Transformed Feature Matrix validated: Zero NaNs, Zero Infs, correct dimensionality.")


## Step 6: Verify Preprocessing Pipeline Idempotency
An idempotent pipeline guarantees that for any input $X$:
$$\text{transform}(X)_{t_1} \equiv \text{transform}(X)_{t_2}$$
Executing the transformation multiple times must yield strictly zero numerical drift.


In [ ]:
idempotency_result = verify_pipeline_idempotency(preprocessor, X_clean, y_clean)

print("=== IDEMPOTENCY AUDIT REPORT ===")
for k, v in idempotency_result.items():
    print(f"{k:<30}: {v}")

assert idempotency_result["is_idempotent"], "Fatal Error: Pipeline failed idempotency test!"
print("
>>> ALL TESTS PASSED: Preprocessing pipeline is strictly idempotent and reproducible.")
